# Phase 6: Exacerbation Label Creation

## Project: ExacerbAlert

### 1. Objective

The objective of this phase is to create supervised learning target labels by aligning documented COPD exacerbation events with the engineered participant-date feature dataset.

The first prediction task will estimate whether a participant experiences at least one documented exacerbation (Moderate or Severe) within the next seven calendar days.

### 2. Input Data

The following project outputs will be used:

- `final_feature_dataset.parquet` — engineered participant-date features generated during Phase 4.
- Exacerbation event records — the 39 documented events imported during Phase 5.
- Participant monitoring dates — derived from the available cleaned sensor observations and the participant-date timeline.

### 3. Prediction Target

The target variable will be named:

`exacerbation_within_7d`

It represents whether at least one documented Moderate or Severe exacerbation occurs within seven calendar days after the feature date.

### 4. Prediction Window

For a feature date \(d\), the prediction window covers:

\[
(d,\ d+7\text{ days}]
\]

The feature date itself is excluded. The seven subsequent calendar dates are included.

This ensures that the target represents a future event rather than an event already occurring on the feature date.

### 5. Label Definitions

| Label | Definition |
|---|---|
| 1 | At least one documented Moderate or Severe exacerbation occurs within the next seven days. |
| 0 | No documented exacerbation occurs within the next seven days, and the complete prediction window is observable. |
| NaN | The complete prediction window is not observable and no positive event is documented within it. |

Known positive events take precedence over incomplete follow-up.

### 6. Event Severity

Both Moderate and Severe events will be treated as positive outcomes for the initial binary prediction task.

Original event severity information will be retained for subsequent severity-specific analysis.

### 7. Event and Participant Alignment

Events will be linked to engineered features using the common participant identifier.

The source exacerbation records contain `Kit-id`, which corresponds to the sensor dataset's `Participant_ID`. The database import process also maps these identifiers to database-generated patient IDs.

Event dates will be interpreted using the UTC date convention established during database import.

### 8. Incomplete Follow-up

A participant-date record must not automatically receive a negative label when the complete seven-day prediction window is not observable.

Participant-specific observation endpoints will be investigated before assigning negative labels.

The last available sensor observation will be treated as an operational observation endpoint, not as definitive evidence of complete clinical event ascertainment.

### 9. Data Leakage Prevention

- Only historical and current-date features may be used as model inputs.
- Future exacerbation records may be used to construct target labels only.
- Event severity, event dates and other future outcome information must not be included as predictive features.
- Feature records and labels must be aligned using participant identifiers and dates.
- Unknown outcomes must not be converted to negative labels.

### 10. Expected Output

The expected output is a participant-date dataset containing:

- `Participant_ID`
- `Date`
- `exacerbation_within_7d`
- The engineered feature columns
- Relevant label-observability and audit information

The resulting dataset will undergo validation of event alignment, class distribution, incomplete follow-up and duplicate participant-date keys before being used for model preparation.

### 11. Project Limitation

These labels represent documented events in the available research dataset. They do not establish that every unrecorded period was clinically event-free, and the resulting model is a research prototype rather than a clinically validated diagnostic or prediction system.

## Step 2: Load and Inspect Feature Data and Exacerbation Events

### Objective

Load the engineered participant-date feature dataset and the original exacerbation event records. Inspect their structure, date ranges, participant coverage and identifier consistency before constructing the supervised-learning target.

### Data sources

1. `data/processed/features/final_feature_dataset.parquet` — engineered daily features, identified by `Participant_ID` and `Date`.
2. Original `exacerbations.csv` files — documented events, containing `Date`, `PatientId`, `Kit-id` and `Type`.

### Processing approach

- Load the final feature dataset without modifying it.
- Read the original event CSV files and retain the participant Kit ID from each file's folder structure.
- Convert feature dates and event dates to UTC-aware timestamps.
- Check required columns, missing values, duplicate event records and severity values.
- Compare participant identifiers between the two datasets.
- Compare event dates with each participant's available feature-data period.

### Important outcome-labeling limitation

An absent exacerbation file does not establish that a participant experienced no exacerbation. Therefore, participants without an event file must not automatically receive negative labels. Negative-label eligibility will be assessed separately, taking the available follow-up period and event-record coverage into account.

### Expected result

A validated feature dataset and a combined event dataset, with aggregate checks showing participant coverage, event counts, severity distribution, date ranges and event records occurring after the available feature-data period.

No labels are created in this step.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Locate the project directory
# --------------------------------------------------

current_dir = Path.cwd()

if (current_dir / "data").exists():
    PROJECT_DIR = current_dir
elif (current_dir.parent / "data").exists():
    PROJECT_DIR = current_dir.parent
else:
    raise FileNotFoundError(
        "Could not locate the project directory containing the data folder."
    )

FEATURES_DIR = PROJECT_DIR / "data" / "processed" / "features"

RAW_INDEX_DIR = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "TOLIFE_COPD_Dataset"
    / "index"
)

FEATURE_FILE = FEATURES_DIR / "final_feature_dataset.parquet"

print("PROJECT DIRECTORY:", PROJECT_DIR)
print("FEATURE FILE EXISTS:", FEATURE_FILE.exists())
print("RAW INDEX DIRECTORY EXISTS:", RAW_INDEX_DIR.exists())


# --------------------------------------------------
# 2. Load the engineered feature dataset
# --------------------------------------------------

if not FEATURE_FILE.exists():
    raise FileNotFoundError(
        f"Feature dataset not found: {FEATURE_FILE}"
    )

features = pd.read_parquet(FEATURE_FILE)

required_feature_columns = ["Participant_ID", "Date"]

missing_feature_columns = [
    col for col in required_feature_columns
    if col not in features.columns
]

if missing_feature_columns:
    raise ValueError(
        f"Missing required feature columns: {missing_feature_columns}"
    )

# Convert dates to UTC-aware timestamps
features["Date"] = pd.to_datetime(
    features["Date"],
    utc=True,
    errors="coerce"
)

features["Participant_ID"] = (
    features["Participant_ID"].astype("string").str.strip()
)

print("\nFEATURE DATASET INSPECTION")
print("=" * 50)
print("Rows:", len(features))
print("Columns:", len(features.columns))
print("Unique participants:", features["Participant_ID"].nunique())
print("Date range:", features["Date"].min(), "to", features["Date"].max())
print("Missing participant IDs:", features["Participant_ID"].isna().sum())
print("Missing dates:", features["Date"].isna().sum())

print(
    "Duplicate participant-date keys:",
    features.duplicated(
        subset=["Participant_ID", "Date"]
    ).sum()
)


# --------------------------------------------------
# 3. Find and load original exacerbation event files
# --------------------------------------------------

event_files = sorted(
    RAW_INDEX_DIR.rglob("exacerbations.csv")
)

print("\nEVENT FILES FOUND:", len(event_files))

if not event_files:
    raise FileNotFoundError(
        "No exacerbations.csv files were found. "
        "Check the raw dataset directory."
    )

event_frames = []

for file in event_files:

    df = pd.read_csv(file)

    required_event_columns = [
        "Date",
        "PatientId",
        "Kit-id",
        "Type"
    ]

    missing_columns = [
        col for col in required_event_columns
        if col not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"{file.name} is missing columns: {missing_columns}"
        )

    # Participant ID comes from the source folder name
    df["Participant_ID"] = file.parent.name

    # Keep the source file for traceability
    df["source_file"] = str(file.relative_to(PROJECT_DIR))

    event_frames.append(df)


# Combine all event records
events = pd.concat(
    event_frames,
    ignore_index=True
)

events["Participant_ID"] = (
    events["Participant_ID"].astype("string").str.strip()
)

events["Type"] = events["Type"].astype("string").str.strip()

events["event_date"] = pd.to_datetime(
    events["Date"],
    utc=True,
    errors="coerce"
)


# --------------------------------------------------
# 4. Validate event records
# --------------------------------------------------

valid_severities = ["Moderate", "Severe"]

invalid_severity = (
    ~events["Type"].isin(valid_severities)
    & events["Type"].notna()
)

exact_duplicate_columns = [
    "Date",
    "PatientId",
    "Kit-id",
    "Type"
]

exact_duplicate_rows = events.duplicated(
    subset=exact_duplicate_columns,
    keep=False
)

print("\nEVENT DATASET INSPECTION")
print("=" * 50)
print("Total event records:", len(events))
print("Participants with event files:", len(event_files))
print("Unique participants with events:", events["Participant_ID"].nunique())

print(
    "Event date range:",
    events["event_date"].min(),
    "to",
    events["event_date"].max()
)

print("Missing event dates:", events["event_date"].isna().sum())
print("Missing participant IDs:", events["Participant_ID"].isna().sum())
print("Missing severity:", events["Type"].isna().sum())
print("Invalid severity records:", invalid_severity.sum())
print("Rows involved in exact duplicate events:", exact_duplicate_rows.sum())

print("\nSEVERITY DISTRIBUTION")
print(events["Type"].value_counts(dropna=False))


# --------------------------------------------------
# 5. Compare participant coverage
# --------------------------------------------------

feature_participants = set(
    features["Participant_ID"].dropna().unique()
)

event_participants = set(
    events["Participant_ID"].dropna().unique()
)

participants_with_features_and_event_files = (
    feature_participants & event_participants
)

participants_without_event_files = (
    feature_participants - event_participants
)

event_participants_without_features = (
    event_participants - feature_participants
)

print("\nPARTICIPANT COVERAGE")
print("=" * 50)
print("Participants with features:", len(feature_participants))
print("Participants with event files:", len(event_participants))
print(
    "Participants present in both:",
    len(participants_with_features_and_event_files)
)
print(
    "Participants without event files:",
    len(participants_without_event_files)
)
print(
    "Event participants without feature data:",
    len(event_participants_without_features)
)


# --------------------------------------------------
# 6. Compare event dates with participant feature endpoints
# --------------------------------------------------

participant_endpoints = (
    features.groupby("Participant_ID")["Date"]
    .max()
    .rename("last_feature_date")
    .reset_index()
)

events_with_endpoints = events.merge(
    participant_endpoints,
    on="Participant_ID",
    how="left",
    validate="many_to_one"
)

events_after_endpoint = events_with_endpoints[
    events_with_endpoints["event_date"]
    > events_with_endpoints["last_feature_date"]
]

print("\nEVENTS AFTER LAST AVAILABLE FEATURE DATE")
print("=" * 50)
print("Events after participant feature endpoint:", len(events_after_endpoint))
print(
    "Participants affected:",
    events_after_endpoint["Participant_ID"].nunique()
)


# --------------------------------------------------
# 7. Summary checks
# --------------------------------------------------

print("\nSUMMARY")
print("=" * 50)
print("Feature data loaded:", not features.empty)
print("Event data loaded:", not events.empty)
print("Feature participant-date keys unique:",
      not features.duplicated(
          subset=["Participant_ID", "Date"]
      ).any())
print("All event dates parsed:", events["event_date"].notna().all())
print("All event severities valid:", not invalid_severity.any())

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\91900\Documents\ExacerbAlert"
)

CLEANED_DIR = PROJECT_DIR / "data" / "processed" / "cleaned"

required_files = [
    "BRI.parquet",
    "EI.parquet",
    "MI.parquet",
    "PEI.parquet",
    "PRVI.parquet",
    "SHRI.parquet",
    "SII.parquet",
    "SQI.parquet"
]

print("CLEANED DATASET CHECK")
print("=" * 45)

for filename in required_files:
    path = CLEANED_DIR / filename
    print(f"{filename}: {'FOUND' if path.exists() else 'MISSING'}")

In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

# Project directories
PROJECT_DIR = Path(
    r"C:\Users\91900\Documents\ExacerbAlert"
)

FEATURES_DIR = PROJECT_DIR / "data" / "processed" / "features"

FEATURE_FILE = FEATURES_DIR / "final_feature_dataset.parquet"

print("FEATURE DATASET CHECK")
print("=" * 50)
print("Feature directory exists:", FEATURES_DIR.exists())
print("Final feature file exists:", FEATURE_FILE.exists())

if not FEATURE_FILE.exists():
    raise FileNotFoundError(
        f"Final feature dataset not found: {FEATURE_FILE}"
    )

# Load final feature dataset
features = pd.read_parquet(FEATURE_FILE)

print("\nDATASET OVERVIEW")
print("=" * 50)
print("Rows:", len(features))
print("Columns:", len(features.columns))
print("Unique participants:", features["Participant_ID"].nunique())

print("\nCOLUMN CHECK")
print("=" * 50)
print("Participant_ID present:", "Participant_ID" in features.columns)
print("Date present:", "Date" in features.columns)

print("\nDATE INFORMATION")
print("=" * 50)
print("Date dtype:", features["Date"].dtype)
print("Earliest date:", features["Date"].min())
print("Latest date:", features["Date"].max())

print("\nMISSING KEY VALUES")
print("=" * 50)
print("Missing Participant_ID:", features["Participant_ID"].isna().sum())
print("Missing Date:", features["Date"].isna().sum())

print("\nDUPLICATE PARTICIPANT-DATE ROWS")
print(
    features.duplicated(
        subset=["Participant_ID", "Date"]
    ).sum()
)

print("\nFIRST FIVE ROWS")
display(features.head())


## Step 3: Inspect Exacerbation Event Records

### Objective
Load and inspect the documented exacerbation event records before constructing the 7-day prediction target.

### Why this step is necessary
The feature dataset contains daily observations for each participant. To create supervised-learning labels, these observations must be compared against the participant's documented exacerbation dates.

We must first understand the event-file structure, available columns, participant identifiers, event dates, and severity categories.

### Planned target
The target variable is `exacerbation_within_7d`:

- `1`: At least one documented Moderate or Severe exacerbation occurs within the next seven days.
- `0`: No documented exacerbation occurs within the next seven days, and the complete outcome window is observable.
- `NaN`: The outcome cannot be reliably determined because sufficient follow-up or event documentation is unavailable.

### Important data limitation
The absence of an exacerbation record does not necessarily prove that a participant experienced no exacerbation. Participants without an event file must not automatically be assigned negative labels.

### Validation requirements
Before label creation, inspect:

1. Number and location of event files.
2. Column names and sample records.
3. Participant identifier consistency.
4. Event dates and severity categories.
5. Duplicate or invalid event records.

Event information will be used to construct the target, not as a predictor feature.

In [ ]:

# STEP 3A: Locate and inspect exacerbation event files

from pathlib import Path
import pandas as pd

# Location of the raw dataset's index directory
RAW_INDEX_DIR = PROJECT_DIR / "data" / "raw" / "TOLIFE_COPD_Dataset" / "index"

print("RAW EVENT DIRECTORY")
print("=" * 50)
print("Directory exists:", RAW_INDEX_DIR.exists())

if not RAW_INDEX_DIR.exists():
    raise FileNotFoundError(
        f"Raw index directory not found: {RAW_INDEX_DIR}"
    )

# Find exacerbation CSV files
event_files = sorted(
    path for path in RAW_INDEX_DIR.rglob("*.csv")
    if "exacerbation" in path.name.lower()
)

print("\nEVENT FILE INVENTORY")
print("=" * 50)
print("Number of event files:", len(event_files))

for path in event_files:
    print("Participant folder:", path.parent.name)
    print("File:", path.name)

# Inspect the first event file
if len(event_files) == 0:
    raise FileNotFoundError("No exacerbation CSV files were found.")

sample_event_file = event_files[0]
sample_events = pd.read_csv(sample_event_file)

print("\nSAMPLE EVENT FILE")
print("=" * 50)
print("File:", sample_event_file)
print("Columns:", sample_events.columns.tolist())
print("Rows:", len(sample_events))

display(sample_events.head(10))


## Step 3: Loading and Auditing Exacerbation Event Records

### Objective
Load all available participant-level exacerbation event files and consolidate them into one event dataset for label creation.

### Procedure
1. Recursively locate all `exacerbations.csv` files in the raw dataset.
2. Load the records and retain the participant folder name as a source identifier.
3. Convert event dates into UTC datetime values.
4. Check required columns, missing values, severity categories, duplicate records, and participant ID consistency.
5. Summarize the number of events by severity and participant.

### Why this is necessary
The event records will be used to construct the supervised learning target. Incorrect dates, participant mappings, or duplicate events could lead to incorrect labels.

### Important dataset limitation
The dataset contains event files for only 24 participants. The absence of an event file for another participant must not automatically be interpreted as proof that the participant experienced no exacerbations. Negative labels will therefore require careful consideration of event documentation and outcome follow-up.

### Interpretation
The audit results will establish the quality and coverage of the available event records. No prediction labels are assigned in this step.

In [ ]:

# STEP 3: Load and audit all exacerbation event records

from pathlib import Path
import pandas as pd
import numpy as np

# Locate raw event directory
EVENT_DIR = (
    PROJECT_DIR
    / "data"
    / "raw"
    / "TOLIFE_COPD_Dataset"
    / "index"
)

# Find all exacerbation files
event_files = sorted(EVENT_DIR.rglob("exacerbations.csv"))

print("EVENT FILE AUDIT")
print("=" * 55)
print("Number of event files:", len(event_files))

# Load all event files
event_frames = []

for file in event_files:

    df = pd.read_csv(file)

    # Record the participant folder and source file
    df["Source_Participant_Folder"] = file.parent.name
    df["Source_File"] = str(file)

    event_frames.append(df)

if not event_frames:
    raise FileNotFoundError("No exacerbation event files found.")

# Combine all event records
events = pd.concat(event_frames, ignore_index=True)

print("\nCOMBINED EVENT DATASET")
print("=" * 55)
print("Total event records:", len(events))
print("Unique Kit-ids:", events["Kit-id"].nunique())
print("Unique participant folders:", events["Source_Participant_Folder"].nunique())

# Check required columns
required_columns = ["Date", "PatientId", "Kit-id", "Type"]

missing_columns = [
    col for col in required_columns
    if col not in events.columns
]

print("\nREQUIRED COLUMN CHECK")
print("=" * 55)
print("Missing required columns:", missing_columns)

if missing_columns:
    raise ValueError(f"Missing required event columns: {missing_columns}")

# Preserve original date and convert to UTC
events["Original_Date"] = events["Date"]
events["Date"] = pd.to_datetime(
    events["Date"],
    errors="coerce",
    utc=True
)

# Missing-value audit
print("\nMISSING VALUE AUDIT")
print("=" * 55)
print(events[required_columns].isna().sum())

# Date validation
print("\nDATE VALIDATION")
print("=" * 55)
print("Invalid dates:", events["Date"].isna().sum())
print("Earliest event:", events["Date"].min())
print("Latest event:", events["Date"].max())

# Severity distribution
print("\nSEVERITY DISTRIBUTION")
print("=" * 55)
print(events["Type"].value_counts(dropna=False))

# Participant mapping validation
folder_mismatches = (
    events["Kit-id"].astype("string").str.strip()
    != events["Source_Participant_Folder"].astype("string").str.strip()
)

print("\nPARTICIPANT FOLDER VALIDATION")
print("=" * 55)
print("Folder and Kit-id mismatches:", folder_mismatches.sum())

if folder_mismatches.any():
    display(
        events.loc[
            folder_mismatches,
            ["Source_Participant_Folder", "Kit-id", "PatientId"]
        ].drop_duplicates()
    )

# Duplicate event validation
duplicate_columns = ["Kit-id", "Date", "Type"]

duplicate_count = events.duplicated(
    subset=duplicate_columns,
    keep=False
).sum()

print("\nDUPLICATE EVENT VALIDATION")
print("=" * 55)
print("Rows involved in duplicate event records:", duplicate_count)

# Events per participant
print("\nEVENTS PER PARTICIPANT")
print("=" * 55)

events_per_participant = (
    events.groupby("Kit-id")
    .agg(
        event_count=("Date", "size"),
        unique_event_dates=("Date", "nunique"),
        patient_id_count=("PatientId", "nunique")
    )
    .sort_index()
)

display(events_per_participant)

# Final sample
print("\nCOMBINED EVENT DATASET SAMPLE")
display(events.head(10))


## Step 4: Participant Observation Period and Event Alignment

### Objective
Compare each participant's available feature observation period with their documented exacerbation event dates.

### Procedure
1. Calculate the first and last available feature dates for each participant.
2. Identify participants with and without an exacerbation event file.
3. Compare event dates with each participant's observation period.
4. Identify events occurring after the last available feature date or before the first available feature date.

### Why this is necessary
A seven-day prediction label requires a clearly defined observation date and outcome window. If the full outcome window is not observable, a negative label cannot safely be assigned.

### Interpretation
Events after the last sensor observation are not automatically invalid. They may still provide positive outcomes for earlier prediction dates. However, negative outcomes require adequate follow-up and event documentation.

The last available sensor observation is an operational endpoint for this analysis. It does not prove that clinical event surveillance ended on that date.

In [ ]:

# STEP 4: Compare participant observation periods with event dates

# Calculate each participant's observation period
participant_periods = (
    features.groupby("Participant_ID")
    .agg(
        first_feature_date=("Date", "min"),
        last_feature_date=("Date", "max"),
        feature_day_count=("Date", "nunique")
    )
    .reset_index()
)

# Identify participants with an event file
event_file_participants = set(
    events["Kit-id"].dropna().astype(str).unique()
)

participant_periods["has_event_file"] = (
    participant_periods["Participant_ID"]
    .isin(event_file_participants)
)

# Merge each event with its participant's observation period
event_alignment = events.merge(
    participant_periods,
    left_on="Kit-id",
    right_on="Participant_ID",
    how="left",
    validate="many_to_one"
)

# Classify event position relative to feature observation period
event_alignment["event_position"] = np.select(
    [
        event_alignment["Date"] < event_alignment["first_feature_date"],
        event_alignment["Date"] > event_alignment["last_feature_date"]
    ],
    [
        "Before first feature date",
        "After last feature date"
    ],
    default="Within feature observation period"
)

print("PARTICIPANT OBSERVATION PERIOD")
print("=" * 60)
print("Participants:", len(participant_periods))
print("Participants with event files:",
      participant_periods["has_event_file"].sum())
print("Participants without event files:",
      (~participant_periods["has_event_file"]).sum())

print("\nOBSERVATION DATE RANGE")
print("=" * 60)
print("Earliest first feature date:",
      participant_periods["first_feature_date"].min())
print("Latest last feature date:",
      participant_periods["last_feature_date"].max())

print("\nEVENT ALIGNMENT")
print("=" * 60)
print(event_alignment["event_position"].value_counts())

print("\nEVENT ALIGNMENT BY SEVERITY")
print("=" * 60)
display(
    pd.crosstab(
        event_alignment["event_position"],
        event_alignment["Type"],
        margins=True
    )
)

print("\nEVENTS AFTER LAST FEATURE DATE")
print("=" * 60)

events_after_endpoint = event_alignment[
    event_alignment["event_position"] == "After last feature date"
]

display(
    events_after_endpoint[
        [
            "Kit-id",
            "Date",
            "Type",
            "last_feature_date"
        ]
    ].sort_values(["Kit-id", "Date"])
)

print("\nPARTICIPANTS WITHOUT EVENT FILES")
print("=" * 60)

display(
    participant_periods.loc[
        ~participant_periods["has_event_file"],
        ["Participant_ID", "first_feature_date", "last_feature_date"]
    ].sort_values("Participant_ID")
)

print("\nPARTICIPANT OBSERVATION SUMMARY")
display(participant_periods.head(10))


## Step 5: Assessing Events Within the Seven-Day Prediction Horizon

### Objective
Determine which documented exacerbation events can be linked to available feature dates under the seven-day prediction horizon.

### Procedure
For each event occurring after a participant's last feature date, calculate the number of days between the last feature date and the event date.

An event may generate a positive label for an available feature date if:

\[
d < e \leq d + 7 \text{ days}
\]

where \(d\) is the feature date and \(e\) is the documented event date.

### Why this is necessary
Events occurring too far beyond the available feature period cannot be predicted using the available feature rows. Events within seven days of the last feature date may still provide valid positive labels.

### Interpretation
This assessment identifies which documented events are usable for the seven-day target. It does not establish that participants without event files experienced no exacerbations.

In [ ]:

# STEP 5: Check event proximity to the last feature date

# Calculate days between last feature date and event date
event_alignment["days_after_last_feature"] = (
    event_alignment["Date"] -
    event_alignment["last_feature_date"]
).dt.days

# Events occurring after the last feature date
events_after_endpoint = event_alignment[
    event_alignment["days_after_last_feature"] > 0
].copy()

# Events occurring within seven days after the last feature date
events_within_7d_after_endpoint = events_after_endpoint[
    events_after_endpoint["days_after_last_feature"].between(1, 7)
].copy()

# Events occurring more than seven days after the last feature date
events_beyond_7d_after_endpoint = events_after_endpoint[
    events_after_endpoint["days_after_last_feature"] > 7
].copy()

print("EVENT USABILITY FOR 7-DAY PREDICTION")
print("=" * 60)

print("Total documented events:", len(events))
print("Events after last feature date:", len(events_after_endpoint))
print(
    "Events within 7 days after last feature date:",
    len(events_within_7d_after_endpoint)
)
print(
    "Events more than 7 days after last feature date:",
    len(events_beyond_7d_after_endpoint)
)

print("\nEVENTS WITHIN 7 DAYS AFTER ENDPOINT")
print("=" * 60)

display(
    events_within_7d_after_endpoint[
        [
            "Kit-id",
            "Date",
            "Type",
            "last_feature_date",
            "days_after_last_feature"
        ]
    ].sort_values(["Kit-id", "Date"])
)

print("\nEVENTS MORE THAN 7 DAYS AFTER ENDPOINT")
print("=" * 60)

display(
    events_beyond_7d_after_endpoint[
        [
            "Kit-id",
            "Date",
            "Type",
            "last_feature_date",
            "days_after_last_feature"
        ]
    ].sort_values(["Kit-id", "Date"])
)

# Count available feature dates that can predict each event
candidate_counts = []

for _, event in events.iterrows():

    participant_dates = features.loc[
        features["Participant_ID"] == event["Kit-id"],
        "Date"
    ]

    candidate_dates = participant_dates[
        (participant_dates < event["Date"]) &
        (participant_dates >= event["Date"] - pd.Timedelta(days=7))
    ]

    candidate_counts.append({
        "Kit-id": event["Kit-id"],
        "Event_Date": event["Date"],
        "Severity": event["Type"],
        "Candidate_Feature_Dates": len(candidate_dates)
    })

event_candidate_summary = pd.DataFrame(candidate_counts)

print("\nAVAILABLE FEATURE DATES PER EVENT")
print("=" * 60)
display(event_candidate_summary)

print(
    "\nEvents with at least one candidate feature date:",
    (event_candidate_summary["Candidate_Feature_Dates"] > 0).sum()
)

print(
    "Events with no candidate feature date:",
    (event_candidate_summary["Candidate_Feature_Dates"] == 0).sum()
)


## Step 6: Creating the Seven-Day Exacerbation Target

### Objective
Create a binary target indicating whether a participant has a documented Moderate or Severe exacerbation within the next seven days.

### Target definition

The target variable is `exacerbation_within_7d`.

For participant \(p\) and feature date \(d\):

\[
Y_{p,d} =
\begin{cases}
1 & \text{if a documented event occurs in } (d,d+7] \\
0 & \text{if no event occurs and full follow-up is available} \\
NaN & \text{if the outcome is unknown}
\end{cases}
\]

### Label assignment rules

**Positive (1)**
- At least one documented Moderate or Severe event occurs after the feature date and within the next seven days.
- A known positive event takes precedence even if the complete seven-day observation window extends beyond the last feature date.

**Negative (0)**
- The participant has an available exacerbation event file.
- No documented event occurs within the prediction window.
- The complete seven-day window ends on or before the participant's last available feature date.

**Unknown (NaN)**
- The participant has no available exacerbation event file and no known positive event.
- The complete seven-day outcome window is not available and no known positive event occurs within it.

### Important methodological decisions
- Both Moderate and Severe events count as positive.
- The feature date itself is excluded from the prediction window because daily features may contain measurements from the entire day.
- Date-only event records are interpreted as UTC dates.
- No-event labels mean no event was documented in the available event file, not proof that no clinical exacerbation occurred.
- Unknown labels are retained as missing values and must not be converted to zero.

### Research limitation
Event-file availability does not independently establish complete clinical event surveillance. Negative labels therefore depend on the assumption that the available participant event records adequately capture documented events during the relevant follow-up period.

The target is intended for research and prototype development, not clinical diagnosis.


# STEP 6: Create seven-day exacerbation labels

PREDICTION_HORIZON = 7

# Make sure dates are UTC datetime values
features["Date"] = pd.to_datetime(
    features["Date"],
    utc=True
)

events["Date"] = pd.to_datetime(
    events["Date"],
    utc=True
)

# Last available feature date for each participant
last_dates = (
    features.groupby("Participant_ID")["Date"]
    .max()
    .to_dict()
)

# Participants with available event files
event_file_participants = set(
    events["Kit-id"].unique()
)

# Event records grouped by participant
event_groups = {
    participant: group.sort_values("Date").reset_index(drop=True)
    for participant, group in events.groupby("Kit-id")
}

# Create a separate label table
labels = features[["Participant_ID", "Date"]].copy()

labels["exacerbation_within_7d"] = np.nan
labels["label_status"] = None
labels["matched_event_date"] = pd.NaT
labels["matched_event_severity"] = None

# Assign labels participant by participant
for participant, group in labels.groupby("Participant_ID"):

    event_group = event_groups.get(participant)

    last_feature_date = last_dates[participant]

    # Get documented event dates for this participant
    if event_group is not None:
        participant_events = event_group.sort_values("Date")
    else:
        participant_events = pd.DataFrame()

    for idx in group.index:

        feature_date = labels.at[idx, "Date"]

        window_end = (
            feature_date +
            pd.Timedelta(days=PREDICTION_HORIZON)
        )

        # Check for documented events in (feature_date, window_end]
        if not participant_events.empty:

            matching_events = participant_events[
                (participant_events["Date"] > feature_date) &
                (participant_events["Date"] <= window_end)
            ]

        else:
            matching_events = pd.DataFrame()

        # Rule 1: A documented event makes the label positive
        if not matching_events.empty:

            matched_event = matching_events.iloc[0]

            labels.at[idx, "exacerbation_within_7d"] = 1
            labels.at[idx, "label_status"] = "positive_documented_event"
            labels.at[idx, "matched_event_date"] = matched_event["Date"]
            labels.at[idx, "matched_event_severity"] = matched_event["Type"]

        # Rule 2: Without an event file, outcome is unknown
        elif participant not in event_file_participants:

            labels.at[idx, "label_status"] = "unknown_no_event_file"

        # Rule 3: No event, but full follow-up is unavailable
        elif window_end > last_feature_date:

            labels.at[idx, "label_status"] = "unknown_incomplete_followup"

        # Rule 4: No documented event and full follow-up available
        else:

            labels.at[idx, "exacerbation_within_7d"] = 0
            labels.at[idx, "label_status"] = "negative_full_followup"

# Display label summary
print("SEVEN-DAY LABEL GENERATION")
print("=" * 60)

print("Total feature rows:", len(labels))
print(
    "Positive labels:",
    (labels["exacerbation_within_7d"] == 1).sum()
)
print(
    "Negative labels:",
    (labels["exacerbation_within_7d"] == 0).sum()
)
print(
    "Unknown labels:",
    labels["exacerbation_within_7d"].isna().sum()
)

print("\nLABEL STATUS BREAKDOWN")
print("=" * 60)
print(labels["label_status"].value_counts(dropna=False))

print("\nTARGET DISTRIBUTION")
print("=" * 60)
print(labels["exacerbation_within_7d"].value_counts(dropna=False))

print("\nSAMPLE LABEL RECORDS")
display(labels.head(10))


In [ ]:

# STEP 6: Create seven-day exacerbation labels

PREDICTION_HORIZON = 7

# Make sure dates are UTC datetime values
features["Date"] = pd.to_datetime(
    features["Date"],
    utc=True
)

events["Date"] = pd.to_datetime(
    events["Date"],
    utc=True
)

# Last available feature date for each participant
last_dates = (
    features.groupby("Participant_ID")["Date"]
    .max()
    .to_dict()
)

# Participants with available event files
event_file_participants = set(
    events["Kit-id"].unique()
)

# Event records grouped by participant
event_groups = {
    participant: group.sort_values("Date").reset_index(drop=True)
    for participant, group in events.groupby("Kit-id")
}

labels = features[["Participant_ID", "Date"]].copy()

labels["exacerbation_within_7d"] = np.nan
labels["label_status"] = None

# Initialize as timezone-aware UTC datetime
labels["matched_event_date"] = pd.Series(
    pd.NaT,
    index=labels.index,
    dtype="datetime64[ns, UTC]"
)

labels["matched_event_severity"] = None

# Assign labels participant by participant
for participant, group in labels.groupby("Participant_ID"):

    event_group = event_groups.get(participant)

    last_feature_date = last_dates[participant]

    # Get documented event dates for this participant
    if event_group is not None:
        participant_events = event_group.sort_values("Date")
    else:
        participant_events = pd.DataFrame()

    for idx in group.index:

        feature_date = labels.at[idx, "Date"]

        window_end = (
            feature_date +
            pd.Timedelta(days=PREDICTION_HORIZON)
        )

        # Check for documented events in (feature_date, window_end]
        if not participant_events.empty:

            matching_events = participant_events[
                (participant_events["Date"] > feature_date) &
                (participant_events["Date"] <= window_end)
            ]

        else:
            matching_events = pd.DataFrame()

        # Rule 1: A documented event makes the label positive
        if not matching_events.empty:

            matched_event = matching_events.iloc[0]

            labels.at[idx, "exacerbation_within_7d"] = 1
            labels.at[idx, "label_status"] = "positive_documented_event"
            labels.at[idx, "matched_event_date"] = matched_event["Date"]
            labels.at[idx, "matched_event_severity"] = matched_event["Type"]

        # Rule 2: Without an event file, outcome is unknown
        elif participant not in event_file_participants:

            labels.at[idx, "label_status"] = "unknown_no_event_file"

        # Rule 3: No event, but full follow-up is unavailable
        elif window_end > last_feature_date:

            labels.at[idx, "label_status"] = "unknown_incomplete_followup"

        # Rule 4: No documented event and full follow-up available
        else:

            labels.at[idx, "exacerbation_within_7d"] = 0
            labels.at[idx, "label_status"] = "negative_full_followup"

# Display label summary
print("SEVEN-DAY LABEL GENERATION")
print("=" * 60)

print("Total feature rows:", len(labels))
print(
    "Positive labels:",
    (labels["exacerbation_within_7d"] == 1).sum()
)
print(
    "Negative labels:",
    (labels["exacerbation_within_7d"] == 0).sum()
)
print(
    "Unknown labels:",
    labels["exacerbation_within_7d"].isna().sum()
)

print("\nLABEL STATUS BREAKDOWN")
print("=" * 60)
print(labels["label_status"].value_counts(dropna=False))

print("\nTARGET DISTRIBUTION")
print("=" * 60)
print(labels["exacerbation_within_7d"].value_counts(dropna=False))

print("\nSAMPLE LABEL RECORDS")
display(labels.head(10))


### Step 8: Validation of Seven-Day Exacerbation Labels

**Objective**

Validate the generated `exacerbation_within_7d` target and its supporting label-status fields before saving the label dataset.

**Validation checks**

1. Confirm that the total number of label records equals the number of feature records.
2. Confirm that each participant-date combination appears only once.
3. Verify that every positive label has a documented event strictly after the feature date and no later than seven days after it.
4. Verify that every negative label has no documented event in the prediction window and has a complete seven-day follow-up period according to the participant's last available feature date.
5. Verify that unknown labels are correctly classified as either having no event file or having incomplete follow-up.
6. Confirm that matched event severities are valid documented categories: Moderate or Severe.

**Important interpretation**

A negative label means that no event was documented in the available event file during a fully observable seven-day window. It does not prove that the participant experienced no clinical exacerbation.

Similarly, an unknown label is not a negative label. Unknown records should be retained for audit purposes but excluded from supervised model training.

**Scope**

These checks validate the internal consistency of the generated labels against the available feature and event records. They do not establish that all clinical exacerbations were captured by the dataset.

In [ ]:

import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Prepare dates and participant-level information
# --------------------------------------------------

labels["Date"] = pd.to_datetime(labels["Date"], utc=True)
labels["matched_event_date"] = pd.to_datetime(
    labels["matched_event_date"], utc=True
)

events["Date"] = pd.to_datetime(events["Date"], utc=True)

last_feature_dates = (
    features.groupby("Participant_ID")["Date"]
    .max()
    .to_dict()
)

event_file_participants = set(events["Kit-id"].astype(str))

# Ensure participant identifiers are comparable
labels["Participant_ID"] = labels["Participant_ID"].astype(str)
events["Kit-id"] = events["Kit-id"].astype(str)

event_groups = {
    participant: group
    for participant, group in events.groupby("Kit-id")
}

# --------------------------------------------------
# 2. Basic structural validation
# --------------------------------------------------

print("BASIC VALIDATION")
print("=" * 60)

print("Feature rows:", len(features))
print("Label rows:", len(labels))

print(
    "Duplicate participant-date rows:",
    labels.duplicated(["Participant_ID", "Date"]).sum()
)

print(
    "Missing participant IDs:",
    labels["Participant_ID"].isna().sum()
)

print(
    "Missing feature dates:",
    labels["Date"].isna().sum()
)

print(
    "Label values:",
    labels["exacerbation_within_7d"].value_counts(dropna=False).to_dict()
)

# --------------------------------------------------
# 3. Validate every label against event records
# --------------------------------------------------

positive_errors = []
negative_errors = []
unknown_errors = []

for idx, row in labels.iterrows():

    participant = row["Participant_ID"]
    feature_date = row["Date"]
    window_end = feature_date + pd.Timedelta(days=7)

    label_value = row["exacerbation_within_7d"]
    status = row["label_status"]

    participant_events = event_groups.get(
        participant,
        pd.DataFrame(columns=events.columns)
    )

    matching_events = participant_events[
        (participant_events["Date"] > feature_date) &
        (participant_events["Date"] <= window_end)
    ]

    last_date = last_feature_dates[participant]

    # Positive label checks
    if label_value == 1:

        matched_date = row["matched_event_date"]
        matched_severity = row["matched_event_severity"]

        if status != "positive_documented_event":
            positive_errors.append((idx, "Incorrect status"))

        if pd.isna(matched_date):
            positive_errors.append((idx, "Missing matched event date"))

        elif not (feature_date < matched_date <= window_end):
            positive_errors.append((idx, "Event outside seven-day window"))

        if matching_events.empty:
            positive_errors.append((idx, "No matching event found"))

        elif not (participant_events["Date"].eq(matched_date) &
                  participant_events["Type"].eq(matched_severity)).any():
            positive_errors.append((idx, "Matched event not found in source"))

        if matched_severity not in ["Moderate", "Severe"]:
            positive_errors.append((idx, "Invalid event severity"))

    # Negative label checks
    elif label_value == 0:

        if status != "negative_full_followup":
            negative_errors.append((idx, "Incorrect status"))

        if participant not in event_file_participants:
            negative_errors.append((idx, "Participant has no event file"))

        if not matching_events.empty:
            negative_errors.append((idx, "Event exists in prediction window"))

        if window_end > last_date:
            negative_errors.append((idx, "Incomplete seven-day follow-up"))

        if pd.notna(row["matched_event_date"]):
            negative_errors.append((idx, "Unexpected matched event date"))

    # Unknown label checks
    elif pd.isna(label_value):

        if participant not in event_file_participants:

            if status != "unknown_no_event_file":
                unknown_errors.append((idx, "Incorrect no-event-file status"))

        else:

            if status != "unknown_incomplete_followup":
                unknown_errors.append((idx, "Incorrect incomplete-follow-up status"))

            if window_end <= last_date:
                unknown_errors.append((idx, "Follow-up was actually complete"))

            if not matching_events.empty:
                unknown_errors.append((idx, "Known event was not labelled positive"))

    else:
        unknown_errors.append((idx, "Unexpected target value"))

# --------------------------------------------------
# 4. Print validation results
# --------------------------------------------------

print("\nLABEL VALIDATION RESULTS")
print("=" * 60)

print("Positive validation errors:", len(positive_errors))
print("Negative validation errors:", len(negative_errors))
print("Unknown validation errors:", len(unknown_errors))

print("\nSAMPLE ERRORS")

print("Positive:", positive_errors[:10])
print("Negative:", negative_errors[:10])
print("Unknown:", unknown_errors[:10])

all_errors = (
    positive_errors +
    negative_errors +
    unknown_errors
)

print("\nOVERALL RESULT")
print("=" * 60)

if len(all_errors) == 0:
    print("PASS: All label validation checks passed.")
else:
    print("REVIEW REQUIRED: Some label validation checks failed.")


### Step 9: Saving the Validated Seven-Day Exacerbation Labels

**Objective**

Save the validated seven-day exacerbation target dataset separately from the engineered feature dataset.

**Output files**

1. `exacerbation_labels_7d.parquet` – primary structured label dataset.
2. `exacerbation_labels_7d_audit.csv` – human-readable audit copy.

**Columns saved**

| Column | Description |
|---|---|
| `Participant_ID` | Participant identifier |
| `Date` | Feature observation date |
| `exacerbation_within_7d` | Target: 1, 0, or NaN |
| `label_status` | Explanation of the label classification |
| `matched_event_date` | Date of the matched documented event, if positive |
| `matched_event_severity` | Moderate or Severe, if positive |

**Storage location**

`data/processed/labels/`

**Validation**

After saving, the Parquet file will be loaded again and checked for:
- Correct row and column counts.
- Correct participant and date information.
- No duplicate participant-date combinations.
- Preserved target distribution.
- Preserved UTC datetime information.

**Important data-handling note**

These files contain participant-level health-related information and event details. They must remain in secure project storage and must not be committed to a public GitHub repository.

In [ ]:

from pathlib import Path
import pandas as pd

# --------------------------------------------------
# 1. Define output directory
# --------------------------------------------------

current_dir = Path.cwd()

if current_dir.name == "notebooks":
    project_root = current_dir.parent
else:
    project_root = current_dir

LABELS_DIR = project_root / "data" / "processed" / "labels"
LABELS_DIR.mkdir(parents=True, exist_ok=True)

parquet_path = LABELS_DIR / "exacerbation_labels_7d.parquet"
csv_path = LABELS_DIR / "exacerbation_labels_7d_audit.csv"

# --------------------------------------------------
# 2. Select final label columns
# --------------------------------------------------

label_columns = [
    "Participant_ID",
    "Date",
    "exacerbation_within_7d",
    "label_status",
    "matched_event_date",
    "matched_event_severity"
]

final_labels = labels[label_columns].copy()

# --------------------------------------------------
# 3. Save both formats
# --------------------------------------------------

final_labels.to_parquet(parquet_path, index=False)
final_labels.to_csv(csv_path, index=False)

print("LABEL DATASETS SAVED")
print("=" * 60)

print("Parquet:", parquet_path.resolve())
print("CSV:", csv_path.resolve())

# --------------------------------------------------
# 4. Read back and verify Parquet
# --------------------------------------------------

saved_labels = pd.read_parquet(parquet_path)

print("\nREAD-BACK VALIDATION")
print("=" * 60)

print("Original rows:", len(final_labels))
print("Saved rows:", len(saved_labels))

print("Original columns:", len(final_labels.columns))
print("Saved columns:", len(saved_labels.columns))

print(
    "Duplicate participant-date rows:",
    saved_labels.duplicated(
        ["Participant_ID", "Date"]
    ).sum()
)

print(
    "Target distribution:",
    saved_labels["exacerbation_within_7d"]
    .value_counts(dropna=False)
    .to_dict()
)

print("Date dtype:", saved_labels["Date"].dtype)
print(
    "Matched event date dtype:",
    saved_labels["matched_event_date"].dtype
)

# --------------------------------------------------
# 5. Final assertions
# --------------------------------------------------

assert len(saved_labels) == len(final_labels)
assert list(saved_labels.columns) == label_columns

assert saved_labels["Participant_ID"].notna().all()
assert saved_labels["Date"].notna().all()

assert not saved_labels.duplicated(
    ["Participant_ID", "Date"]
).any()


# Compare positive and negative counts separately
assert (
    saved_labels["exacerbation_within_7d"].eq(1).sum()
    ==
    final_labels["exacerbation_within_7d"].eq(1).sum()
)

assert (
    saved_labels["exacerbation_within_7d"].eq(0).sum()
    ==
    final_labels["exacerbation_within_7d"].eq(0).sum()
)

# Compare missing/unknown counts separately
assert (
    saved_labels["exacerbation_within_7d"].isna().sum()
    ==
    final_labels["exacerbation_within_7d"].isna().sum()
)

print("Target distribution comparison: PASS")


assert saved_labels["Date"].dt.tz is not None
assert str(saved_labels["Date"].dt.tz) == "UTC"

assert saved_labels["matched_event_date"].dt.tz is not None
assert str(saved_labels["matched_event_date"].dt.tz) == "UTC"

print("Timezone validation: PASS")

print("\nPASS: Label dataset saved and verified successfully.")
